In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pipeline.src.python.config as cfg
from loguru import logger
import pandas as pd
import polars as pl
import numpy as np
from bertopic import BERTopic

pd.set_option('display.max_colwidth', None)

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 ## Science News topic over time

### Carico i dati 

In [3]:
sn_documents = pl.read_parquet('/home/banfi/TETYS/pipeline/src/python/data/raw/science_news_pipeline_data.parquet')

### Carico il modello

In [4]:
model  = BERTopic.load('/home/banfi/TETYS/pipeline/src/python/models/science_news/model_0.309.safetensors',embedding_model=cfg.EMBEDDING_MODEL)

### Prendo solo mese e anno per il binning della data

In [5]:
sn_documents = sn_documents.with_columns(
    pl.col('publicationDate').dt.truncate('1mo').alias('timestamp'),
)

### Creo una chiave id per matchare il timestamp per dare lo stesso ordine di training

In [6]:
sn_documents_id_and_timestamp = sn_documents.select(['id','timestamp'])

### Carico i dati utilizzati durante il training

In [7]:
sn_embeddings_data = np.load('/home/banfi/TETYS/pipeline/src/python/data/interim/embeddings/science_news/science_news_embeddings_aggregation_with_lemma.npz',allow_pickle = True)

### Creo il dateset che utilizzerò per la visualizzazione

In [8]:
sn_text = pl.DataFrame({'id': sn_embeddings_data['id'],
                        'text': sn_embeddings_data['text'],
                        'topic': model.topics_},
                        )

### Creo l'indice per avere lo stesso ordine iniziale (pre-join)

In [9]:
sn_text = sn_text.with_row_index('index')

### Faccio la join

In [10]:
sn_timeseries_df  = sn_text.join(sn_documents_id_and_timestamp,on='id').sort(by='index',descending=False)

### Dati parziali (non tutti i periodi temporali hanno un valore numerico per i topic, questo porta ad un grafico non equo)

In [11]:
partial_sn_timeseries_data = sn_timeseries_df.group_by(['topic','timestamp']).agg(
    pl.len().alias('occurrence')
).sort(by=['timestamp','topic'],descending=False)


In [12]:
sn_timeseries_df.filter ((pl.col('topic') == 28)).sort(by='timestamp').filter(pl.col('timestamp').dt.year() == 2010 ).select('text').to_pandas()

,text
0,"Researchers found that the relative abundance of proteins and mRNA varies significantly between individual cells, challenging previous assumptions about their relationship. A study in *Science* revealed that mRNA and proteins exist on different time scales, with mRNA degrading quickly and proteins persisting longer, leading to discrepancies in measurement. This highlights the complexity of molecular interactions within cells."


### Calcolo tutto il periodo temporale

In [13]:
dates = pl.date_range(
    start= sn_timeseries_df.select(pl.col("timestamp").min()).item(),
    end=  sn_timeseries_df.select(pl.col("timestamp").max()).item(),
    interval= '1mo',
    eager = True
)

### Prendo tutti i topic

In [14]:
topics = ( 
    sn_timeseries_df.select(
        pl.col('topic').unique().sort()).to_series()
)


### Calcolo la matrice - TOPIC x PERIODO

In [15]:
grid = (
    pl.DataFrame({'topic':topics}).join(pl.DataFrame({'timestamp':dates}),how='cross')
)

### Riempio i buchi nei dati

In [16]:
grid.select(pl.col('timestamp')).head()

timestamp
date
2001-02-01
2001-03-01
2001-04-01
2001-05-01
2001-06-01


In [17]:
partial_sn_timeseries_data = partial_sn_timeseries_data.with_columns(
        pl.col("timestamp").dt.date().alias("timestamp")
)

In [18]:
partial_sn_timeseries_data.select(pl.col('timestamp')).head()

timestamp
date
2001-02-01
2001-02-01
2001-02-01
2001-02-01
2001-03-01


In [19]:
complete_sn_timeseries_data  = (
    grid.join(partial_sn_timeseries_data,on=['topic','timestamp'],how='left')
    .with_columns(pl.col('occurrence').fill_null(0))
    .sort(by=['timestamp','topic'],descending=False)
)

In [20]:
complete_sn_timeseries_pandas_data = complete_sn_timeseries_data.to_pandas()

In [26]:
topics_hierarchy = model.hierarchical_topics(sn_embeddings_data['text'])

100%|██████████| 28/28 [00:00<00:00, 291.61it/s]


## Display temporal hirarchy topics

In [126]:
max_distance = 2

In [127]:
topics_group = topics_hierarchy[ topics_hierarchy['Distance'] <= max_distance].sort_values(['Parent_ID'],ascending=True)['Topics'].to_list()

In [128]:
topics_group


[[0, 3],
 [2, 7],
 [1, 11],
 [2, 7, 20],
 [0, 3, 4],
 [5, 15],
 [0, 3, 4, 21],
 [5, 6, 15],
 [2, 7, 16, 20],
 [9, 27],
 [8, 10],
 [1, 11, 13],
 [1, 8, 10, 11, 13],
 [12, 14],
 [17, 23],
 [24, 28],
 [18, 19],
 [12, 14, 24, 28],
 [2, 7, 16, 20, 22],
 [1, 8, 9, 10, 11, 13, 27],
 [17, 18, 19, 23],
 [1, 8, 9, 10, 11, 13, 25, 27],
 [5, 6, 15, 26],
 [0, 2, 3, 4, 7, 16, 20, 21, 22],
 [5, 6, 15, 17, 18, 19, 23, 26],
 [0, 1, 2, 3, 4, 7, 8, 9, 10, 11, 13, 16, 20, 21, 22, 25, 27],
 [5, 6, 12, 14, 15, 17, 18, 19, 23, 24, 26, 28],
 [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28]]

In [129]:
unique_topics_group = list(topics_group)  # copia + assicura lista
to_remove = set()

for i in range(len(topics_group) - 1):
    for j in range(i + 1, len(topics_group)):
        if set(topics_group[i]).issubset(set(topics_group[j])):
            print(f'{topics_group[i]} -> {topics_group[j]}')
            to_remove.add(i)
            continue

unique_topics_group = [x for k, x in enumerate(topics_group) if k not in to_remove]

print(topics_group)
print(unique_topics_group)


[0, 3] -> [0, 3, 4]
[0, 3] -> [0, 3, 4, 21]
[0, 3] -> [0, 2, 3, 4, 7, 16, 20, 21, 22]
[0, 3] -> [0, 1, 2, 3, 4, 7, 8, 9, 10, 11, 13, 16, 20, 21, 22, 25, 27]
[0, 3] -> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
[2, 7] -> [2, 7, 20]
[2, 7] -> [2, 7, 16, 20]
[2, 7] -> [2, 7, 16, 20, 22]
[2, 7] -> [0, 2, 3, 4, 7, 16, 20, 21, 22]
[2, 7] -> [0, 1, 2, 3, 4, 7, 8, 9, 10, 11, 13, 16, 20, 21, 22, 25, 27]
[2, 7] -> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
[1, 11] -> [1, 11, 13]
[1, 11] -> [1, 8, 10, 11, 13]
[1, 11] -> [1, 8, 9, 10, 11, 13, 27]
[1, 11] -> [1, 8, 9, 10, 11, 13, 25, 27]
[1, 11] -> [0, 1, 2, 3, 4, 7, 8, 9, 10, 11, 13, 16, 20, 21, 22, 25, 27]
[1, 11] -> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
[2, 7, 20] -> [2, 7, 16, 20]
[2, 7, 20] -> [2, 7, 16, 20, 22]
[2, 7, 20] -> [0, 2, 3, 4, 7, 16, 20, 

In [130]:
topics = [topic for topics_list in unique_topics_group for topic in topics_list]

In [131]:
topics_set  = set(topics)

In [132]:
unique_topics = set(model.topics_)
n_topics = len(unique_topics) -1 if -1 in unique_topics else len(unique_topics)

In [133]:
total_topics = set([i for i in range(n_topics)])

In [134]:
topics_to_add = total_topics - topics_set

In [135]:
for topic in topics_to_add:
    unique_topics_group.append([topic])

In [136]:
unique_topics_group

[[0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28]]

In [137]:
unique_topics_group

[[0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28]]

In [138]:
import pandas as pd


group_map = {}
for g in unique_topics_group:
    label = "_".join(map(str, g))
    for t in g:
        group_map[t] = label

# 2) crea la colonna "topic_group" (se topic non mappato -> resta se stesso)
df = complete_sn_timeseries_pandas_data.copy()
df["topic_group"] = df["topic"].map(group_map).fillna(df["topic"].astype(str))

# 3) raggruppa per data e topic_group e somma le occorrenze
out = (df.groupby(["timestamp", "topic_group"], as_index=False)["occurrence"]
         .sum()
         .sort_values(["timestamp", "topic_group"]))

out


,timestamp,topic_group,occurrence
0,2001-02-01,-1,1
1,2001-02-01,0_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16_17_18_19_20_21_22_23_24_25_26_27_28,4
2,2001-03-01,-1,1
3,2001-03-01,0_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16_17_18_19_20_21_22_23_24_25_26_27_28,0
4,2001-04-01,-1,0
...,...,...,...
593,2025-10-01,0_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16_17_18_19_20_21_22_23_24_25_26_27_28,0
594,2025-11-01,-1,0
595,2025-11-01,0_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16_17_18_19_20_21_22_23_24_25_26_27_28,2
596,2025-12-01,-1,1


In [139]:
import plotly.express as px

fig = px.line(
    out,
    x="timestamp",
    y="occurrence",
    color="topic_group",
    markers=True,
    title="Andamento temporale dei topic"
)

fig.update_layout(
    xaxis_title="Tempo",
    yaxis_title="Occorrenze",
    legend_title="topic_group"
)

fig.show()

In [22]:
model.visualize_hierarchy().show()

In [140]:
import plotly.express as px

fig = px.line(
    complete_sn_timeseries_pandas_data,
    x="timestamp",
    y="occurrence",
    color="topic",
    markers=True,
    title="Andamento temporale dei topic"
)

fig.update_layout(
    xaxis_title="Time",
    yaxis_title="No.Articles",
    legend_title="Topic"
)

fig.show()


In [120]:
import plotly.express as px

fig = px.line(
    complete_sn_timeseries_pandas_data,
    x="timestamp",
    y="occurrence",
    color="topic",
    markers=True,
    title="Andamento temporale dei topic",
    line_shape='spline'  # Aggiunto qui
)

fig.update_layout(
    xaxis_title="Tempo",
    yaxis_title="Occorrenze",
    legend_title="Topic"
)

fig.show()

## Bertopic visualization

In [56]:
dates = (
    sn_timeseries_df
    .select(pl.col("timestamp").unique().sort())
    .to_series()
    .to_list()
)


In [58]:
topics_over_time = model.topics_over_time(sn_timeseries_df['text'].to_list(),
                                          sn_timeseries_df['timestamp'].to_list(),
                                          nr_bins=len(dates)
                                          )

2026-01-29 11:01:28,545 - BERTopic - WARNING: There are more than 100 unique timestamps (i.e., 274) which significantly slows down the application. Consider setting `nr_bins` to a value lower than 100 to speed up calculation. 
274it [00:07, 35.61it/s]


0    2001-02-01
1    2001-03-01
2    2001-04-01
3    2001-05-01
4    2001-06-01
5    2001-07-01
6    2001-08-01
7    2001-09-01
8    2001-10-01
9    2001-11-01
10   2001-12-01
11   2002-01-01
12   2002-02-01
13   2002-03-01
14   2002-04-01
15   2002-05-01
16   2002-06-01
17   2002-07-01
18   2002-08-01
19   2002-09-01
20   2002-10-01
21   2002-11-01
22   2002-12-01
23   2003-01-01
Name: literal, dtype: datetime64[ms]

In [62]:
fig = model.visualize_topics_over_time(topics_over_time)

In [63]:
fig

In [64]:
sn_timeseries_df

index,id,text,timestamp
u32,str,str,datetime[μs]
0,"""https://www.sciencenews.org/ar…","""Among patients who recovered f…",2020-04-01 00:00:00
1,"""https://www.sciencenews.org/ar…","""Drugs used to treat high blood…",2020-04-01 00:00:00
2,"""https://www.sciencenews.org/ar…","""With more men than women devel…",2020-04-01 00:00:00
3,"""https://www.sciencenews.org/ar…","""Nivedita Lakhera, a nurse at O…",2020-04-01 00:00:00
4,"""https://www.sciencenews.org/ar…","""For millions of years, the Chi…",2020-04-01 00:00:00
…,…,…,…
1157,"""https://www.sciencenews.org/ar…","""When I was a kid, my best frie…",2005-09-01 00:00:00
1158,"""https://www.sciencenews.org/ar…","""Bryan Christie's ""Milestones"" …",2012-09-01 00:00:00
1159,"""https://www.sciencenews.org/ar…","""Whether you're planning a beac…",2006-08-01 00:00:00
